# Car Sales Prediction Using Linear Regression

**Project type:** Regression | **Tools:** Python, Pandas, NumPy, Seaborn, Matplotlib, Statsmodels, Scikit-learn

## Project Overview

This case study explores a car-sales dataset and develops a Linear Regression model to understand and predict car sales. The workflow follows the original analysis: exploratory data analysis (EDA), data preparation, categorical encoding, target transformation, feature selection, multicollinearity assessment, and model building.

> **Dataset requirement:** Place `Car_sales.csv` in the `data/` folder beside this notebook before running it.

## 1. Import Libraries

Import the libraries used for data handling, visualization, statistical analysis, feature selection, model fitting, and evaluation.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import f_regression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from patsy import dmatrices

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

## 2. Load the Dataset

Use a relative file path so the notebook can run on another computer after the dataset is placed in the expected folder.

In [ ]:
from pathlib import Path

DATA_PATH = Path("data") / "Car_sales.csv"
cars = pd.read_csv(DATA_PATH)

cars.head()

## 3. Data Understanding and Exploratory Analysis

### 3.1 Dataset structure

In [ ]:
print(f"Number of observations: {cars.shape[0]:,}")
print(f"Number of variables: {cars.shape[1]:,}")

cars.info()

### 3.2 Categorical variables and cardinality

In [ ]:
categorical_columns = cars.select_dtypes(include=["object", "category"]).columns
cars.loc[:, categorical_columns].head()

In [ ]:
for column in categorical_columns:
    print(f"{column}: {cars[column].nunique(dropna=True)} unique values")
    display(cars[column].value_counts(dropna=False).head(15))

### 3.3 Descriptive statistics

In [ ]:
cars.describe(include="all").T

### 3.4 Missing values

In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": cars.isna().sum(),
    "Missing_Percentage": (cars.isna().mean() * 100).round(2),
})
missing_summary.sort_values("Missing_Percentage", ascending=False)

### 3.5 Initial business hypotheses

The original analysis considered these expected relationships:

- Higher horsepower may be associated with larger engines, higher prices, and larger vehicle dimensions.
- Higher fuel efficiency may be associated with lower fuel consumption.

These are hypotheses to investigate in the data, not conclusions until supported by analysis.

### 3.6 Correlation among numeric variables

In [ ]:
numeric_columns = cars.select_dtypes(include=np.number).columns
numeric_correlations = cars[numeric_columns].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(numeric_correlations, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix — Numeric Variables")
plt.tight_layout()
plt.show()

## 4. Data Quality Review

The following summary function reports missingness, cardinality, descriptive statistics, and IQR-based outlier bounds for numeric columns.

In [ ]:
def continuous_var_summary(series):
    total = len(series)
    missing = series.isna().sum()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    return pd.Series({
        "dtype": str(series.dtype),
        "cardinality": series.nunique(dropna=True),
        "total_rows": total,
        "non_missing": series.count(),
        "missing": missing,
        "missing_pct": missing / total * 100 if total else np.nan,
        "mean": series.mean(),
        "std": series.std(),
        "min": series.min(),
        "p01": series.quantile(0.01),
        "p25": q1,
        "median": series.median(),
        "p75": q3,
        "p99": series.quantile(0.99),
        "max": series.max(),
        "iqr_lower_bound": q1 - 1.5 * iqr,
        "iqr_upper_bound": q3 + 1.5 * iqr,
    })

numeric_data = cars.select_dtypes(include=np.number).copy()
numeric_data.apply(continuous_var_summary).T

### 4.1 Categorical data summary

In [ ]:
cars.select_dtypes(include=["object", "category"]).describe().T

## 5. Data Preparation

The original workflow treats numeric outliers using the 1st and 99th percentiles, imputes numeric missing values with column means, and one-hot encodes `Manufacturer` and `Vehicle_type`.

**Interpretation note:** percentile clipping changes extreme numeric values. In a production workflow, fit preprocessing rules on the training data only to avoid information leakage. This notebook retains the case-study sequence for transparency.

### 5.1 Treat numeric outliers using percentile clipping

In [ ]:
numeric_data = numeric_data.apply(
    lambda column: column.clip(
        lower=column.quantile(0.01),
        upper=column.quantile(0.99),
    )
)

numeric_data.apply(continuous_var_summary).T

### 5.2 Impute missing numeric values

In [ ]:
numeric_data = numeric_data.apply(lambda column: column.fillna(column.mean()))
numeric_data.isna().sum()

### 5.3 Encode selected categorical variables

The original case study uses `Manufacturer` and `Vehicle_type` to create dummy variables. `drop_first=True` omits one category from each variable to reduce redundant dummy columns.

In [ ]:
categorical_features = cars.loc[:, ["Manufacturer", "Vehicle_type"]].copy()
categorical_features = categorical_features.fillna("Unknown")

categorical_encoded = pd.get_dummies(
    categorical_features,
    columns=["Manufacturer", "Vehicle_type"],
    drop_first=True,
    dtype=int,
)

categorical_encoded = categorical_encoded.rename(
    columns={"Manufacturer_Mercedes-B": "Manufacturer_Mercedes_B"}
)
categorical_encoded.columns.tolist()

### 5.4 Combine numeric and encoded categorical data

In [ ]:
cars_model = pd.concat(
    [numeric_data.reset_index(drop=True), categorical_encoded.reset_index(drop=True)],
    axis=1,
)

cars_model.head()

## 6. Prepare the Target Variable

The target variable is `Sales_in_thousands`. The original analysis examines its distribution and skewness, then applies a natural-log transformation to create `lg_sales_in_thousands`.

A log transformation requires positive target values.

In [ ]:
target = "Sales_in_thousands"

if (cars_model[target] <= 0).any():
    raise ValueError("Log transformation requires all Sales_in_thousands values to be positive.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(cars_model[target], kde=True, ax=axes[0])
axes[0].set_title("Sales in Thousands — Original Scale")
sns.histplot(np.log(cars_model[target]), kde=True, ax=axes[1])
axes[1].set_title("Log-Transformed Sales")
plt.tight_layout()
plt.show()

print("Original target skewness:", cars_model[target].skew())
print("Log-transformed target skewness:", np.log(cars_model[target]).skew())

In [ ]:
cars_model["lg_sales_in_thousands"] = np.log(cars_model[target])
cars_model[["Sales_in_thousands", "lg_sales_in_thousands"]].head()

### 6.1 Check feature relationships and sample size

In [ ]:
correlation_matrix = cars_model.corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, cmap="coolwarm", center=0)
plt.title("Correlation Matrix — Prepared Modeling Data")
plt.tight_layout()
plt.show()

In [ ]:
target_columns = ["Sales_in_thousands", "lg_sales_in_thousands"]
features = [column for column in cars_model.columns if column not in target_columns]

print(f"Number of observations: {len(cars_model):,}")
print(f"Number of candidate features: {len(features):,}")

## 7. Feature Selection

The original case study uses `f_regression` to assess each candidate feature against the log-transformed target. Features with p-values below 0.10 are retained for the next stage.

This is a univariate screening step; it does not, by itself, prove that a feature is useful in a multivariable predictive model.

In [ ]:
f_scores, p_values = f_regression(cars_model[features], cars_model["lg_sales_in_thousands"])

significant_variables = pd.DataFrame({
    "Feature": features,
    "F_Score": f_scores,
    "P_Value": p_values,
}).sort_values("P_Value")

significant_variables

In [ ]:
SIGNIFICANCE_LEVEL = 0.10
selected_features = significant_variables.loc[
    significant_variables["P_Value"] < SIGNIFICANCE_LEVEL, "Feature"
].tolist()

print(f"Selected features (p < {SIGNIFICANCE_LEVEL}): {len(selected_features)}")
selected_features

## 8. Multicollinearity Assessment

Variance Inflation Factor (VIF) is used to assess multicollinearity among the selected features. A high VIF can indicate that a feature is strongly explained by the other predictors.

The intercept is included in the design matrix for model fitting; its VIF is shown for completeness but is not interpreted as a predictor.

In [ ]:
if selected_features:
    vif_formula = "lg_sales_in_thousands ~ " + " + ".join(selected_features)
    y_vif, X_vif = dmatrices(vif_formula, cars_model, return_type="dataframe")

    vif_summary = pd.DataFrame({
        "Feature": X_vif.columns,
        "VIF": [
            variance_inflation_factor(X_vif.values, i)
            for i in range(X_vif.shape[1])
        ],
    })
    vif_summary
else:
    vif_summary = pd.DataFrame(columns=["Feature", "VIF"])
    print("No features met the significance threshold. Review the feature-selection step.")

## 9. Train-Test Split

The original case study uses a 70% training set and a 30% test set with `random_state=10`. The split is performed before fitting the regression models.

In [ ]:
train, test = train_test_split(
    cars_model,
    train_size=0.70,
    test_size=0.30,
    random_state=10,
)

print(f"Training rows: {len(train):,}")
print(f"Test rows: {len(test):,}")

## 10. Model 0 — Baseline Regression

Model 0 uses all candidate features identified before feature reduction and predicts the log-transformed sales target.

In [ ]:
formula_0 = "lg_sales_in_thousands ~ " + " + ".join(features)
model_0 = smf.ols(formula_0, data=train).fit()

print(model_0.summary())

## 11. Model 1 — Regression After Feature Selection

Model 1 uses the features that passed the univariate significance threshold.

In [ ]:
if selected_features:
    formula_1 = "lg_sales_in_thousands ~ " + " + ".join(selected_features)
    model_1 = smf.ols(formula_1, data=train).fit()
    print(model_1.summary())
else:
    model_1 = None
    print("Model 1 was not fitted because no features passed the selection threshold.")

## 12. Model 2 — Business-Guided Feature Reduction

The original case study also explored a manually specified model, selecting variables based on statistical and business considerations. The formula below follows the variables recorded in the original notebook; it may need adjustment if the dataset's generated dummy-column names differ.

In [ ]:
formula_2 = (
    "lg_sales_in_thousands ~ Fuel_efficiency + Horsepower + Length "
    "+ Manufacturer_Audi + Manufacturer_Ford + Manufacturer_Honda "
    "+ Manufacturer_Mercedes_B + Manufacturer_Plymouth + Manufacturer_Porsche "
    "+ Manufacturer_Toyota + Manufacturer_Volvo + Power_perf_factor "
    "+ Price_in_thousands + Vehicle_type_Passenger + Wheelbase"
)

available_formula_columns = set(cars_model.columns)
required_columns = [
    term.strip()
    for term in formula_2.split("~", 1)[1].split("+")
]
missing_formula_columns = [col for col in required_columns if col not in available_formula_columns]

if missing_formula_columns:
    print("Model 2 needs these columns to be reviewed:", missing_formula_columns)
    model_2 = None
else:
    model_2 = smf.ols(formula_2, data=train).fit()
    print(model_2.summary())

## 13. Model Evaluation — Next Step

The original notebook describes model evaluation using MAPE, RMSE, correlation between actual and predicted values, and decile analysis, but does not complete a consistent evaluation section for the final model.

Before publishing, compare the selected model on the held-out test set. Since the target was log-transformed, decide whether to report metrics on the log scale or convert predictions back to the original sales scale. Clearly state which scale is used.

## 14. Key Considerations and Limitations

- This notebook documents a case-study workflow, not a production-ready prediction pipeline.
- Numeric imputation, percentile clipping, and feature selection were performed before the train-test split in the original approach. This can cause information leakage; for a stronger predictive evaluation, these steps should be learned from the training data only.
- The log-transformed target changes how model predictions should be interpreted.
- Final model selection should be supported by test-set metrics and business context, not training R-squared alone.
- The dataset is not included in this notebook. Add it to `data/Car_sales.csv` if you have permission to redistribute it; otherwise provide instructions for obtaining it.

## 15. Project Summary

This case study covers exploratory data analysis, missing-value treatment, percentile-based outlier handling, one-hot encoding, log transformation, univariate feature selection, VIF analysis, train-test splitting, and multiple Ordinary Least Squares regression models.